In this notebook we will implement a simple version of CoOp (Context Optimization) for few-shot adaptation on the Flowers102 dataset.

In [ ]:
#install clip
!pip install ftfy regex tqdm
!pip install openai_clip

In [ ]:
#necessary imports
import torch
import torchvision
import torch.nn as nn
import clip
from torch.nn import functional as F
from tqdm import tqdm
from collections import OrderedDict

We will now create functions to correctly get our data and split it into base and novel classes.

In [ ]:
def get_data(data_dir="./data", transform=None):
    """Load Flowers102 train, validation and test sets.
    Args:
        data_dir (str): Directory where the dataset will be stored.
        transform (torch.Compose)
    Returns:
        tuple: A tuple containing the train, validation, and test sets.
    """
    train = torchvision.datasets.Flowers102(root=data_dir, split="train", download=True, transform=transform)
    val = torchvision.datasets.Flowers102(root=data_dir, split="val", download=True, transform=transform)
    test = torchvision.datasets.Flowers102(root=data_dir, split="test", download=True, transform=transform)
    return train, val, test

In [ ]:
def base_novel_categories(dataset):
    # set returns the unique set of all dataset classes
    all_classes = set(dataset._labels)
    # and let's count them
    num_classes = len(all_classes)

    # here list(range(num_classes)) returns a list from 0 to num_classes - 1
    # then we slice the list in half and generate base and novel category lists
    base_classes = list(range(num_classes))[:num_classes//2]
    novel_classes = list(range(num_classes))[num_classes//2:]
    return base_classes, novel_classes

Let's inspect the classes.

In [ ]:
_, _, tmp_test = get_data()
base_classes, novel_classes = base_novel_categories(tmp_test)
class_names = ["pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
                "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
                "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
                "yellow iris", "globe-flower", "purple coneflower", "peruvian lily", "balloon flower",
                "giant white arum lily", "fire lily", "pincushion flower", "fritillary", "red ginger",
                "grape hyacinth", "corn poppy", "prince of wales feathers", "stemless gentian", "artichoke",
                "sweet william", "carnation", "garden phlox", "love in the mist", "mexican aster",
                "alpine sea holly", "ruby-lipped cattleya", "cape flower", "great masterwort", "siam tulip",
                "lenten rose", "barbeton daisy", "daffodil", "sword lily", "poinsettia", "bolero deep blue",
                "wallflower", "marigold", "buttercup", "oxeye daisy", "common dandelion", "petunia", "wild pansy",
                "primula", "sunflower", "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia",
                "pink-yellow dahlia", "cautleya spicata", "japanese anemone", "black-eyed susan", "silverbush",
                "californian poppy", "osteospermum", "spring crocus", "bearded iris", "windflower", "tree poppy",
                "gazania", "azalea", "water lily", "rose", "thorn apple", "morning glory", "passion flower", "lotus",
                "toad lily", "anthurium", "frangipani", "clematis", "hibiscus", "columbine", "desert-rose",
                "tree mallow", "magnolia", "cyclamen", "watercress", "canna lily", "hippeastrum", "bee balm",
                "ball moss", "foxglove", "bougainvillea", "camellia", "mallow", "mexican petunia", "bromelia",
                "blanket flower", "trumpet creeper", "blackberry lily"]
print("Base Class Names:", [(i, class_names[i]) for i in base_classes])
print("Novel Class Names:", [(i, class_names[i]) for i in novel_classes])

Let's now split the dataset.

In [ ]:
def split_data(dataset, base_classes):
    # these two lists will store the sample indexes
    base_categories_samples = []
    novel_categories_samples = []

    # we create a set of base classes to compute the test below in O(1)
    # this is optional and can be removed
    base_set = set(base_classes)

    # here we iterate over sample labels and also get the correspondent sample index
    for sample_id, label in enumerate(dataset._labels):
        if label in base_set:
            base_categories_samples.append(sample_id)
        else:
            novel_categories_samples.append(sample_id)

    # here we create the dataset subsets
    # the torch Subset is just a wrapper around the dataset
    # it simply stores the subset indexes and the original dataset (your_subset.dataset)
    # when asking for sample i in the subset, torch will look for its original position in the dataset and retrieve it
    # https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset
    base_dataset = torch.utils.data.Subset(dataset, base_categories_samples)
    novel_dataset = torch.utils.data.Subset(dataset, novel_categories_samples)
    return base_dataset, novel_dataset

In [ ]:
def create_remapped_dataset(dataset, selected_classes):
  #TODO: vibecodato, funziona
    """Create a dataset subset with remapped labels.

    Args:
        dataset: Original dataset
        selected_classes: List of class indices to include

    Returns:
        Subset dataset with labels remapped to [0, len(selected_classes)-1]
    """
    # Create mapping from original labels to new labels
    label_map = {old_label: new_label for new_label, old_label in enumerate(selected_classes)}
    selected_set = set(selected_classes)

    # Find samples and create new labels
    selected_samples = []
    new_labels = []

    for sample_id, label in enumerate(dataset._labels):
        if label in selected_set:
            selected_samples.append(sample_id)
            new_labels.append(label_map[label])

    # Create subset
    subset = torch.utils.data.Subset(dataset, selected_samples)

    # Add remapped labels to subset
    subset.remapped_labels = new_labels

    return subset

class RemappedDataset(torch.utils.data.Dataset):
    #TODO: vibecodato, funziona
    """Wrapper dataset that returns remapped labels"""
    def __init__(self, subset_dataset):
        self.dataset = subset_dataset.dataset
        self.indices = subset_dataset.indices
        self.labels = subset_dataset.remapped_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Get original sample
        original_idx = self.indices[idx]
        image, _ = self.dataset[original_idx]  # Ignore original label

        # Return with remapped label
        return image, self.labels[idx]

Let's now load a pretrained CLIP model.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Load the CLIP model and preprocessing transform
clip_model, preprocess = clip.load("ViT-B/16", device=device)
clip_model.eval()  # We won't fine-tune CLIP
for param in clip_model.parameters():
    param.requires_grad = False

# Get the image and text encoders
image_encoder = clip_model.visual
text_encoder = clip_model.encode_text  # Used in inference mode only


Let's prepare the train-test-splits.


In [ ]:
# get the three datasets
train_set, val_set, test_set = get_data(transform=preprocess)

# split classes into base and novel
base_classes, novel_classes = base_novel_categories(train_set)

# split the three datasets
train_base, _ = split_data(train_set, base_classes)
val_base, _ = split_data(val_set, base_classes)
test_base, test_novel = split_data(test_set, base_classes)

Now we will start implementing CoOp.

 The first thing to do is create a **PromptLearner** class. We will randomly initialize the context vector and **class_name** will be at the end of the prompt followed by EOS token. We will use 32 as number of context tokens.

In [ ]:
class PromptLearner(nn.Module):
    def __init__(self, class_names, clip_model, n_ctx=16):
      super().__init__()
      self.class_names = class_names
      self.n_cls = len(class_names)
      self.n_ctx = n_ctx
      self.clip_model = clip_model
      self.tokenizer = clip.tokenize

      # CLIP parameters
      dtype = clip_model.dtype
      ctx_dim = clip_model.ln_final.weight.shape[0]
      self.ctx_dim = ctx_dim
      # Get the device from the clip_model
      self.device = next(clip_model.parameters()).device


      # Random context initialization
      ctx_vectors = torch.empty(n_ctx, ctx_dim, dtype=dtype)
      nn.init.normal_(ctx_vectors, std=0.02)
      prompt_prefix = " ".join(["X"] * n_ctx)

      self.ctx = nn.Parameter(ctx_vectors)  # shared learnable context, to be optimized

      # Use clip.tokenize directly as it doesn't have an encode method
      name_lens = [len(self.tokenizer(name)[0]) for name in class_names]
      prompts = [prompt_prefix + " " + name + "." for name in class_names]

      tokenized_prompts = torch.cat([self.tokenizer(prompt) for prompt in prompts]).to(self.device) # Move to device
      with torch.no_grad():
        embedding = clip_model.token_embedding(tokenized_prompts).type(dtype)

      self.register_buffer("token_prefix", embedding[:, :1, :]) # SOS
      self.register_buffer("token_suffix", embedding[:, 1 + n_ctx :, :]) # CLS, EOS

      self.tokenized_prompts = tokenized_prompts
      self.name_lens = name_lens


    def forward(self):

      prefix = self.token_prefix
      ctx = self.ctx
      if ctx.dim() == 2:
            ctx = ctx.unsqueeze(0).expand(self.n_cls, -1, -1)
      suffix = self.token_suffix

      prompts = torch.cat(
        [
          prefix,
          ctx,
          suffix,
        ],
        dim = 1,
      )

      return prompts

We now have to create a **TextEncoder** class.

In [ ]:
class TextEncoder(nn.Module):
  def __init__(self, clip_model):
    super().__init__()
    self.transformer = clip_model.transformer
    self.positional_embedding = clip_model.positional_embedding
    self.ln_final = clip_model.ln_final
    self.text_projection = clip_model.text_projection  # This was missing!
    self.dtype = clip_model.dtype

  def forward(self, prompts, tokenized_prompts):
    x = prompts + self.positional_embedding.type(self.dtype)
    x = x.permute(1,0,2)  # [seq_len, n_cls, embed_dim]
    x = self.transformer(x)
    x = x.permute(1,0,2)  # [n_cls, seq_len, embed_dim]
    x = self.ln_final(x).type(self.dtype)

    # Take features from EOS embedding
    x = x[torch.arange(x.shape[0]), tokenized_prompts.argmax(dim=-1)] @ self.text_projection

    return x

We are now ready to create out **CustomCLIP** class.

In [ ]:
class CustomCLIP(nn.Module):
  def __init__(self, active_classnames, all_classnames, clip_model, n_ctx=16, alpha=1.0, template = "A photo of a {}, a type of flower.", **kwargs):
    super().__init__()
    self.prompt_learner = PromptLearner(active_classnames, clip_model, n_ctx)
    self.all_classnames = all_classnames
    self.tokenized_prompts = self.prompt_learner.tokenized_prompts
    self.text_encoder = TextEncoder(clip_model)
    self.clip_model = clip_model
    self.image_encoder = clip_model.visual
    self.logit_scale = clip_model.logit_scale
    self.dtype = clip_model.dtype
    self.alpha = alpha
    self.device = next(clip_model.parameters()).device
    self.target = self._compute_target(template=template)

    print(f"Instance of CustomCLIP created with n_ctx={n_ctx} and alpha={alpha}")
  
  def _compute_target(self, template = "A photo of a {}, a type of flower."):
    # Create prompts for each class name (base + novel)
    texts = [template.format(name) for name in self.all_classnames] # [C]

    # Tokenize 
    tokenized = clip.tokenize(texts).to(self.device) # [C, 77]

    # Encode
    embedded = self.clip_model.encode_text(tokenized).type(self.dtype) 
    embedded = embedded / embedded.norm(dim=-1, keepdim=True) # [C, 512]
    
    # Average
    avg = embedded.mean(dim=0)
    avg = avg / avg.norm()

    return avg

  def forward(self, images):

    # Encode images
    image_features = self.image_encoder(images.type(self.dtype))

    # Encode text
    prompts = self.prompt_learner()
    tokenized_prompts = self.tokenized_prompts
    text_features = self.text_encoder(prompts, tokenized_prompts)

    # Normalize
    image_features = image_features / image_features.norm(dim=1, keepdim=True)
    text_features = text_features / text_features.norm(dim=1, keepdim=True)

    # Compute similarity logits
    logit_scale = self.logit_scale.exp()
    logits = logit_scale * image_features @ text_features.t()

    avg_soft = text_features.mean(dim=0)
    avg_soft = avg_soft / avg_soft.norm(dim=0)
    text_loss = 1 - (avg_soft @ self.target)
    text_loss = self.alpha * text_loss

    return logits, text_loss

Let's now create training and evaluation functions.

In [ ]:
def train_one_epoch(model, dataloader, optimizer, scheduler, loss_function, device):
  model.train()
  total_loss = 0.0
  total_correct = 0
  total_samples = 0

  for images, labels in dataloader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    logits, text_loss = model(images)
    loss = loss_function(logits, labels) + text_loss
    loss.backward()
    optimizer.step()
    scheduler.step()

    total_loss += loss.item() * images.size(0)
    total_correct += (logits.argmax(dim=1) == labels).sum().item()
    total_samples += images.size(0)

  avg_loss = total_loss / total_samples
  accuracy = total_correct / total_samples
  return avg_loss, accuracy


def evaluate(model, dataloader, device):
  model.eval()
  total_correct = 0
  total_samples = 0

  with torch.no_grad():
    for images, labels in dataloader:
      images = images.to(device)
      labels = labels.to(device)

      logits, _ = model(images)
      total_correct += (logits.argmax(dim=1) == labels).sum().item()
      total_samples += images.size(0)

  accuracy = total_correct / total_samples
  return accuracy

Now let's set up the training for CoOp on the base classes.

In [ ]:
# Create properly remapped datasets for base classes
train_base_remapped = create_remapped_dataset(train_set, base_classes)
val_base_remapped = create_remapped_dataset(val_set, base_classes)
test_base_remapped = create_remapped_dataset(test_set, base_classes)

# Wrap with RemappedDataset to get correct labels
train_base_dataset = RemappedDataset(train_base_remapped)
val_base_dataset = RemappedDataset(val_base_remapped)
test_base_dataset = RemappedDataset(test_base_remapped)

print(f"Base training samples: {len(train_base_dataset)}")
print(f"Base validation samples: {len(val_base_dataset)}")
print(f"Base test samples: {len(test_base_dataset)}")
print(f"Number of base classes: {len(base_classes)}")

# Get base class names for the model
base_class_names = [class_names[i] for i in base_classes]
print(f"Base class names: {base_class_names[:5]}...")  # Show first 5

In [ ]:
# Create data loaders
batch_size = 1
train_loader = torch.utils.data.DataLoader(
    train_base_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4
)
val_loader = torch.utils.data.DataLoader(
    val_base_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)
test_loader = torch.utils.data.DataLoader(
    test_base_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
n_ctx = 4
alpha = 5.0
novel_class_names = [class_names[i] for i in novel_classes]
# Initialize the CoOp model
coop_model = CustomCLIP(active_classnames=base_class_names, all_classnames=novel_class_names, clip_model=clip_model, n_ctx=n_ctx, alpha=alpha).to(device)

# Set up optimizer - only optimize the context vectors
optimizer = torch.optim.SGD([coop_model.prompt_learner.ctx], lr=0.002, momentum=0.9)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200, eta_min=0.0001)

# Loss function
loss_function = nn.CrossEntropyLoss()

# Training parameters
num_epochs = 5
best_val_acc = 0.0

print(f"Model initialized with {len(base_class_names)} base classes")
print(f"Context dimension: {coop_model.prompt_learner.ctx_dim}")
print(f"Number of context tokens: {coop_model.prompt_learner.n_ctx}")
print(f"Trainable parameters: {sum(p.numel() for p in coop_model.parameters() if p.requires_grad)}")

# Verify only context vectors are trainable
print("\nTrainable parameters:")
for name, param in coop_model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

In [ ]:
# Training loop
print("Starting training...")
training_history = {
    'train_loss': [],
    'train_acc': [],
    'val_acc': []
}

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)

    # Training phase
    train_loss, train_acc = train_one_epoch(coop_model, train_loader, optimizer, scheduler, loss_function, device)

    # Validation phase
    val_acc = evaluate(coop_model, val_loader, device)

    # Save training history
    training_history['train_loss'].append(train_loss)
    training_history['train_acc'].append(train_acc)
    training_history['val_acc'].append(val_acc)

    # Print metrics
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"New best validation accuracy: {best_val_acc:.4f}")
        # Save model checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': coop_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
        }, 'best_coop_model.pth')

print(f"\nTraining completed!")
print(f"Best validation accuracy: {best_val_acc:.4f}")

In [ ]:
# Load best model and evaluate on test set
checkpoint = torch.load('best_coop_model.pth')
coop_model.load_state_dict(checkpoint['model_state_dict'])

test_acc = evaluate(coop_model, test_loader, device)
print(f"Test Accuracy on Base Classes: {test_acc:.4f}")

# Plot training curves
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(training_history['train_loss'], label='Training Loss')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(training_history['train_acc'], label='Training Accuracy')
plt.plot(training_history['val_acc'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Now let's evaluate on novel classes to test generalization
print("\n" + "="*60)
print("EVALUATING ON NOVEL CLASSES")
print("="*60)

# Create novel class datasets with remapped labels
test_novel_remapped = create_remapped_dataset(test_set, novel_classes)
test_novel_dataset = RemappedDataset(test_novel_remapped)

# Get novel class names
novel_class_names = [class_names[i] for i in novel_classes]
print(f"Novel classes: {len(novel_classes)}")
print(f"Novel test samples: {len(test_novel_dataset)}")
print(f"Novel class names: {novel_class_names[:5]}...")

# Create a new model for novel classes (same learned context, different class names)
novel_coop_model = CustomCLIP(active_classnames=novel_class_names, all_classnames=novel_class_names, clip_model=clip_model, n_ctx=n_ctx, alpha=alpha).to(device)

# Load the learned context from the trained model
novel_coop_model.prompt_learner.ctx.data = coop_model.prompt_learner.ctx.data.clone()

# Create data loader for novel classes
test_novel_loader = torch.utils.data.DataLoader(
    test_novel_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

# Evaluate on novel classes
novel_test_acc = evaluate(novel_coop_model, test_novel_loader, device)
print(f"\nTest Accuracy on Novel Classes: {novel_test_acc:.4f}")

# Compare base vs novel performance
print(f"\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)
print(f"Base Classes Test Accuracy: {test_acc:.4f}")
print(f"Novel Classes Test Accuracy: {novel_test_acc:.4f}")
print(f"Harmonic mean: {(2 / (1/test_acc + 1/novel_test_acc)):.4f}")

# This shows how well the learned context generalizes to unseen classes

In [ ]:
# ...place at the top of your notebook...
import sys
import os
sys.path.append(os.path.abspath("../src"))

from hill_climbing import hill_climbing_hyperparameter_search

novel_class_names = [class_names[i] for i in novel_classes]
base_class_names = [class_names[i] for i in base_classes]

hyperparameters = {
    'fixed': {
        'device': 'cuda',
    },
    'tunable': {
        'n_ctx': [4, 8, 12, 16],
        'train_batch_size': [1, 4, 8, 16, 32],
        'val_batch_size': [64],
        'num_epochs': [3, 5, 7, 10],
        'learning_rate': [0.02],
        'momentum': [0.9],
        'alpha': [0.1, 0.5, 1.0, 5.0, 10.0, 20.0],
        'T_max': [20],
        'eta_min': [0.0001],
        'active_classnames': [base_class_names],
        'all_classnames': [novel_class_names],
        'clip_model': [clip_model],
    }
}
initial_config = {
    'fixed': {
        'device': 'cuda',
    },
    'tunable': {
        'n_ctx': 4,
        'train_batch_size': 16,
        'val_batch_size': 64,
        'num_epochs': 5,
        'learning_rate': 0.02,
        'momentum': 0.9,
        'alpha': 0.5,
        'T_max': 20,
        'eta_min': 0.0001,
        'active_classnames': base_class_names,
        'all_classnames': novel_class_names,
        'clip_model': clip_model,
    }
}

model, best_config, best_acc, log_steps = hill_climbing_hyperparameter_search(
    train_dataset=train_base_dataset,
    val_dataset=val_base_dataset,
    initial_config=initial_config,
    hyperparams=hyperparameters,
    loss_function=nn.CrossEntropyLoss(),
    model_class=CustomCLIP,
    optimizer_class=torch.optim.SGD,
    scheduler_class=torch.optim.lr_scheduler.CosineAnnealingLR
)